# 2. Core Gene Selection and Cross-Perturbation Matrices

## What it does

**Core gene selection:** Selects N genes from the quality-filtered DPD ranking as the gene set for network inference. These are the genes whose regulatory relationships we will reconstruct. Selection is by absolute |DPD_stim_norm|, taking equal numbers from the activator and suppressor tails to avoid the network being dominated by one direction.

**Cross-perturbation matrices (pseudobulk):** For each core gene, compute the mean log fold-change expression profile from DE_stats. The cross-perturbation matrix M[i,j] = e_i · e_j is the dot product of the full expression profiles of perturbations i and j. A cosine-normalised version is also computed. These matrices are the input to the pseudobulk BMRA inference in Notebook 3 Section B, and serve as the fast diagnostic baseline while ROLS runs.

## Contents
1. Configuration
2. Load DPD scores and reference vectors
3. Select core genes
4. Build expression matrices from DE_stats
5. UMAP of core gene perturbations
6. Build cross-perturbation matrices
7. Save


### 1. Configuration

In [1]:
import os
print(f'Working directory: {os.getcwd()}')

Working directory: /mnt/R0/Projects/POIAZ/Ilaria/Scripts


In [2]:
# Must match Notebook 1
CONDITION = 'Rest' # 'Rest', 'Stim8hr', 'Stim48hr'
DONORS = ['D1', 'D2', 'D3', 'D4']
donors_tag = '_'.join(DONORS)
run_tag = f'{CONDITION}_{donors_tag}'

# Paths
DATA_DIR = '../Data'
DESTATS_PATH = '../Data/GWCD4i.DE_stats.h5ad'
REF_DIR = '../Results/ref'
BTLA_TIMEPOINT = '4h' # must match Notebook 0
IN_DIR = f'../Results/{CONDITION}' # Notebook 1 outputs
OUT_DIR = f'../Results/{CONDITION}'
os.makedirs(OUT_DIR, exist_ok=True)

# Core gene selection
# N_CORE=200 for pseudobulk pinv stability. For ROLS (Notebook 3), this can be increased
# to 400-500 once the pipeline is validated. Change N_CORE here and rerun this notebook
# before rerunning Notebook 3.
N_CORE = 200
# Select equal numbers from activator and suppressor tails so the network is not
# dominated by one direction. Half from positive DPD_stim_norm, half from negative.
SELECTION_STRATEGY = 'both_tails'  # 'both_tails' or 'abs' (top by absolute value)

# UMAP requires loading the full perturbed cell AnnData files for the core genes.
# Set RUN_UMAP = False to skip and just build the cross-perturbation matrices.
RUN_UMAP = False
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.3
# Set REPLOT_UMAP=True to reload cached adata_umap and replot without recomputing.
REPLOT_UMAP=True

# DE_stats layer and threshold
P_THRESHOLD = 0.05
LAYER_KEY = 'log_fc'

print(f'run_tag: {run_tag}')
print(f'N_CORE: {N_CORE}')
print(f'Strategy: {SELECTION_STRATEGY}')
print(f'Run UMAP: {RUN_UMAP}')
print(f'Out dir: {OUT_DIR}')

run_tag: Rest_D1_D2_D3_D4
N_CORE: 200
Strategy: both_tails
Run UMAP: False
Out dir: ../Results/Rest


In [3]:
import gc
import numpy as np
import pandas as pd
import scipy.sparse
import anndata
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['font.size'] = 11

if RUN_UMAP:
    import scanpy as sc
    print('scanpy loaded for UMAP.')

print('Packages loaded.')

Packages loaded.


### 2. Load DPD Scores and Reference Vectors

In [4]:
dpd_df = pd.read_csv(os.path.join(IN_DIR, f'DPD_ranked_filtered_{run_tag}.csv'))
print(f'Quality-filtered DPD table: {len(dpd_df):,} genes')
print(f'Columns: {dpd_df.columns.tolist()}')
print(f'DPD_stim_norm range: [{dpd_df["DPD_stim_norm"].min():.3f}, {dpd_df["DPD_stim_norm"].max():.3f}]')

Quality-filtered DPD table: 6,017 genes
Columns: ['rank', 'target_contrast_gene_name', 'target_contrast', 'culture_condition', 'n_cells_target', 'n_total_de_genes', 'ontarget_effect_size', 'ontarget_significant', 'offtarget_flag', 'n_sig_genes', 'DPD_stim', 'DPD_stim_norm', 'DPD_btla', 'DPD_btla_norm']
DPD_stim_norm range: [-2.066, 4.768]


In [5]:
# Stimulation vector
stim_vec_df = pd.read_csv(os.path.join(IN_DIR, f'v_stim_{run_tag}.csv'))
stim_sig = stim_vec_df[stim_vec_df['significant'] == True].copy()
stim_genes = stim_sig['ensembl_id'].values
v_stim = stim_sig['logFC_sig'].values.astype(np.float32)
stim_norm = float(np.linalg.norm(v_stim))
print(f'v_stim: {len(v_stim):,} significant genes, norm={stim_norm:.4f}')

# BTLA vector
btla_vec_df = pd.read_csv(os.path.join(REF_DIR, f'v_btla_{BTLA_TIMEPOINT}.csv'))
v_btla_full = btla_vec_df['logFC_btla'].values.astype(np.float32)
btla_norm = float(np.linalg.norm(v_btla_full))
print(f'v_btla: {len(v_btla_full):,} entries ({(v_btla_full != 0).sum():,} non-zero), norm={btla_norm:.4f}')

# DE_stats var names (needed to align vectors to matrix columns)
de_stats = anndata.read_h5ad(DESTATS_PATH, backed='r')
gene_ids_de_stats = de_stats.var_names.values
print(f'DE_stats genes: {len(gene_ids_de_stats):,}')

v_stim: 1,199 significant genes, norm=7.3412
v_btla: 10,282 entries (650 non-zero), norm=18.6903
DE_stats genes: 10,282


### 3. Select Core Genes

Select the N_CORE genes that will form the network nodes. We use the `both_tails` strategy by default: N/2 from the highest positive DPD_stim_norm (strongest activation enhancers) and N/2 from the most negative DPD_stim_norm (strongest activation suppressors). This ensures the network represents both regulatory directions.

The `abs` strategy (top by |DPD_stim_norm|) naturally produces ~75% suppressors because the suppressor tail is longer and more extreme (CD3E reaches -16.8; the max activator is ~+1.5). `both_tails` gives a more balanced network.

In [6]:
if SELECTION_STRATEGY == 'both_tails':
    half = N_CORE // 2
    top = dpd_df.nlargest(half, 'DPD_stim_norm')
    bottom = dpd_df.nsmallest(half, 'DPD_stim_norm')
    core_df = pd.concat([top, bottom]).drop_duplicates(subset='target_contrast_gene_name')
elif SELECTION_STRATEGY == 'abs':
    core_df = dpd_df.reindex(
        dpd_df['DPD_stim_norm'].abs().sort_values(ascending=False).index
    ).head(N_CORE).copy()
else:
    raise ValueError(f'Unknown SELECTION_STRATEGY: {SELECTION_STRATEGY}')

core_genes = core_df['target_contrast_gene_name'].tolist()
core_ensembl = core_df['target_contrast'].tolist()  # Ensembl IDs of the perturbed genes

print(f'Core genes selected: {len(core_genes)}')
print(f'  Activators (DPD_stim > 0): {(core_df["DPD_stim_norm"] > 0).sum()}')
print(f'  Suppressors (DPD_stim < 0): {(core_df["DPD_stim_norm"] < 0).sum()}')
print(f'  DPD_stim_norm range: [{core_df["DPD_stim_norm"].min():.3f}, {core_df["DPD_stim_norm"].max():.3f}]')
print(f'\nTop 10 activators: {core_df.nlargest(10, "DPD_stim_norm")["target_contrast_gene_name"].tolist()}')
print(f'Top 10 suppressors: {core_df.nsmallest(10, "DPD_stim_norm")["target_contrast_gene_name"].tolist()}')

Core genes selected: 200
  Activators (DPD_stim > 0): 100
  Suppressors (DPD_stim < 0): 100
  DPD_stim_norm range: [-2.066, 4.768]

Top 10 activators: ['UBXN1', 'NCKAP1L', 'ST6GALNAC6', 'FAM89B', 'ZMYM2', 'SENP5', 'ABI3', 'ATP2A2', 'CPSF6', 'SMS']
Top 10 suppressors: ['LIG1', 'MED15', 'SMARCA5', 'DHX36', 'TAF13', 'TSC1', 'C11orf54', 'MAEA', 'LEO1', 'ELAVL1']


### 4. Build Expression Matrices from DE_stats

For each core gene, extract the log fold-change profile from DE_stats for this condition. If multiple guide rows exist for the same gene (from different CRISPR guides targeting the same gene), average them. This produces one consensus expression vector per gene.

Two matrices are built:
- `expr_stim`: (N_CORE × |stim_significant_genes|), used for the stim cross-perturbation matrix
- `expr_btla`: (N_CORE × 10,282), full gene space, used for the BTLA cross-perturbation matrix

In [7]:
# Subset DE_stats to the chosen condition and core genes
obs_meta = de_stats.obs.copy()
cond_mask = obs_meta['culture_condition'] == CONDITION
core_mask = obs_meta['target_contrast_gene_name'].isin(core_genes)
combined_mask = cond_mask & core_mask
obs_core = obs_meta[combined_mask].copy()

found_genes = set(obs_core['target_contrast_gene_name'].unique())
missing = set(core_genes) - found_genes
if missing:
    print(f'WARNING: {len(missing)} core genes not found in DE_stats for {CONDITION}: {sorted(missing)}')
    core_genes = [g for g in core_genes if g in found_genes]
    core_df = core_df[core_df['target_contrast_gene_name'].isin(core_genes)]
print(f'Rows in DE_stats matching core genes + condition: {combined_mask.sum():,}')

# Same pattern as Notebook 1 get_dense: subset with boolean mask and read layer directly.
# No .copy() or .to_memory() — those fail because X is stored as null in this file.
DE_core = de_stats[combined_mask]
raw_slice = DE_core.layers[LAYER_KEY]
if scipy.sparse.issparse(raw_slice):
    raw_slice = raw_slice.toarray()
raw_slice = np.asarray(raw_slice, dtype=np.float32)
gene_names_in_slice = obs_core['target_contrast_gene_name'].values
print(f'Loaded slice: {raw_slice.shape}')

# Identify column positions for stim and btla gene subsets
de_to_idx = {g: i for i, g in enumerate(gene_ids_de_stats)}
stim_col_idx = np.array([de_to_idx[g] for g in stim_genes if g in de_to_idx], dtype=np.int32)
stim_genes_used = np.array([g for g in stim_genes if g in de_to_idx])
# For btla: all columns (v_btla_full is already in DE_stats gene order)
print(f'Stim gene columns: {len(stim_col_idx):,}')

# Build per-gene mean expression profiles (average across guides)
expr_stim = np.zeros((len(core_genes), len(stim_col_idx)), dtype=np.float32)
expr_btla = np.zeros((len(core_genes), raw_slice.shape[1]), dtype=np.float32)

for i, gene in enumerate(core_genes):
    rows = raw_slice[gene_names_in_slice == gene]
    expr_stim[i] = rows[:, stim_col_idx].mean(axis=0)
    expr_btla[i] = rows.mean(axis=0)

del raw_slice
gc.collect()
print(f'expr_stim: {expr_stim.shape}  ({expr_stim.nbytes/1024**2:.1f} MB)')
print(f'expr_btla: {expr_btla.shape}  ({expr_btla.nbytes/1024**2:.1f} MB)')

Rows in DE_stats matching core genes + condition: 200
Loaded slice: (200, 10282)
Stim gene columns: 1,199
expr_stim: (200, 1199)  (0.9 MB)
expr_btla: (200, 10282)  (7.8 MB)


### 5. UMAP of Core Gene Perturbations

Load the perturbed single-cell data for the N_CORE selected genes and compute a UMAP.

**What we expect to see:** Strong suppressors should cluster away from NTC-like cells towards a hypo-activated region. Strong activators should cluster in the opposite direction. If the stim and BTLA axes are genuinely orthogonal, genes with high DPD_btla_norm should shift along a different direction than genes with high DPD_stim_norm.

**Data source:** The full assigned-guide h5ad files. We load only cells belonging to the core gene perturbations for this condition, plus a sample of NTC cells as a reference anchor. We do not load the full >100 GB files, we filter in backed mode.

**Design note:** We use scanpy for UMAP here. Standard preprocessing: normalise to 10,000 counts per cell, log1p, select highly variable genes (2,000), PCA (50 components), neighbours (n_neighbors=15), UMAP.

In [8]:
UMAP_CACHE_PATH = os.path.join(OUT_DIR, f'umap_adata_{run_tag}.h5ad')

if not RUN_UMAP:
    print('RUN_UMAP=False, skipping.')
elif os.path.exists(UMAP_CACHE_PATH):
    print(f'Loading cached adata_umap from {UMAP_CACHE_PATH}')
    adata_umap = anndata.read_h5ad(UMAP_CACHE_PATH)
    print(f'Loaded: {adata_umap.shape}')

    # Annotate cells with DPD scores from the core_df table
    dpd_lookup = core_df.set_index('target_contrast_gene_name')[['DPD_stim_norm', 'DPD_btla_norm']]
    adata_umap.obs['DPD_stim_norm'] = adata_umap.obs['perturbed_gene_name'].map(
        dpd_lookup['DPD_stim_norm']).fillna(0).astype(float)
    adata_umap.obs['DPD_btla_norm'] = adata_umap.obs['perturbed_gene_name'].map(
        dpd_lookup['DPD_btla_norm']).fillna(0).astype(float)
    adata_umap.obs['is_ntc'] = adata_umap.obs['guide_type'] == 'non-targeting'

    # Plot 1: coloured by DPD_stim_norm
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    coords = adata_umap.obsm['X_umap']
    stim_vals = adata_umap.obs['DPD_stim_norm'].values
    btla_vals = adata_umap.obs['DPD_btla_norm'].values

    for ax, vals, label, cmap in [
        (axes[0], stim_vals, 'DPD_stim_norm', 'RdBu_r'),
        (axes[1], btla_vals, f'DPD_btla_norm ({BTLA_TIMEPOINT})', 'PuOr_r')
    ]:
        vmax = np.percentile(np.abs(vals[vals != 0]), 95) if (vals != 0).any() else 1
        sc_plot = ax.scatter(coords[:, 0], coords[:, 1],
                             c=vals, cmap=cmap, vmin=-vmax, vmax=vmax,
                             s=2, alpha=0.6, rasterized=True)
        plt.colorbar(sc_plot, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xlabel('UMAP 1', fontsize=11)
        ax.set_ylabel('UMAP 2', fontsize=11)
        ax.set_title(f'UMAP coloured by {label}', fontsize=12, fontweight='bold')

    plt.suptitle(f'Core gene perturbations: {CONDITION}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    p = os.path.join(OUT_DIR, f'umap_core_perturbations_{run_tag}.png')
    plt.savefig(p, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: {p}')

    # Plot 2: coloured by perturbation identity (highlight top 10 each tail)
    top10_act = core_df.nlargest(10, 'DPD_stim_norm')['target_contrast_gene_name'].tolist()
    top10_sup = core_df.nsmallest(10, 'DPD_stim_norm')['target_contrast_gene_name'].tolist()
    highlight = set(top10_act + top10_sup)
    label_col = adata_umap.obs['perturbed_gene_name'].apply(
        lambda x: x if x in highlight else ('NTC' if x == 'non-targeting' else 'other'))
    adata_umap.obs['highlight_label'] = label_col

    fig, ax = plt.subplots(figsize=(12, 9))
    palette = {'NTC': '#cccccc', 'other': '#dddddd'}
    for gene in top10_act:
        palette[gene] = '#d62728'
    for gene in top10_sup:
        palette[gene] = '#1f77b4'

    for label in ['NTC', 'other'] + top10_act + top10_sup:
        mask = adata_umap.obs['highlight_label'] == label
        size = 2 if label in ['NTC', 'other'] else 12
        alpha = 0.3 if label in ['NTC', 'other'] else 0.9
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=palette[label], s=size, alpha=alpha,
                   label=label if label not in ['NTC', 'other'] else None,
                   rasterized=True)
    ax.legend(markerscale=4, fontsize=8, ncol=2, loc='best')
    ax.set_xlabel('UMAP 1', fontsize=11)
    ax.set_ylabel('UMAP 2', fontsize=11)
    ax.set_title(f'Top 10 activators (red) and suppressors (blue), {CONDITION}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    p = os.path.join(OUT_DIR, f'umap_top_perturbations_labelled_{run_tag}.png')
    plt.savefig(p, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: {p}')
else:
    print('Loading perturbed cells for core genes...')
    # ... all the existing data loading + sc.pp + sc.tl.umap code ...
    donor_adatas = []
    for donor in DONORS:
        fpath = os.path.join(DATA_DIR, f'{donor}_{CONDITION}.assigned_guide.h5ad')
        tmp = anndata.read_h5ad(fpath, backed='r')
        # Keep cells that are either NTC or belong to a core gene perturbation
        keep = (
            (tmp.obs['guide_type'] == 'non-targeting') |
            (tmp.obs['perturbed_gene_name'].isin(core_genes))
        )
        # Subsample NTC to avoid it dominating the UMAP
        ntc_idx = np.where(tmp.obs['guide_type'] == 'non-targeting')[0]
        pert_idx = np.where(
            (tmp.obs['guide_type'] != 'non-targeting') &
            (tmp.obs['perturbed_gene_name'].isin(core_genes))
        )[0]
        n_ntc_sample = min(len(ntc_idx), 5000)  # cap NTC at 5,000 cells per donor
        rng = np.random.default_rng(42)
        ntc_sample = rng.choice(ntc_idx, size=n_ntc_sample, replace=False)
        keep_idx = np.concatenate([ntc_sample, pert_idx])
        sub = tmp[keep_idx].to_memory()
        tmp.file.close()
        donor_adatas.append(sub)
        print(f'  {donor}: {sub.n_obs:,} cells ({len(pert_idx):,} perturbed + {n_ntc_sample:,} NTC)')

    adata_umap = anndata.concat(donor_adatas, label='donor', keys=DONORS)
    del donor_adatas
    gc.collect()
    print(f'Combined: {adata_umap.shape}')

    # Standard single-cell preprocessing
    sc.pp.normalize_total(adata_umap, target_sum=1e4)
    sc.pp.log1p(adata_umap)
    sc.pp.highly_variable_genes(adata_umap, n_top_genes=2000, flavor='seurat')
    sc.pp.pca(adata_umap, n_comps=50, mask_var='highly_variable')
    sc.pp.neighbors(adata_umap, n_neighbors=UMAP_N_NEIGHBORS, n_pcs=50)
    sc.tl.umap(adata_umap, min_dist=UMAP_MIN_DIST)
    print('UMAP computed.')
    # Save UMAP AnnData object
    adata_umap.write_h5ad(UMAP_CACHE_PATH)
    print(f'adata_umap cached to {UMAP_CACHE_PATH}')

    # Annotate cells with DPD scores from the core_df table
    dpd_lookup = core_df.set_index('target_contrast_gene_name')[['DPD_stim_norm', 'DPD_btla_norm']]
    adata_umap.obs['DPD_stim_norm'] = adata_umap.obs['perturbed_gene_name'].map(
        dpd_lookup['DPD_stim_norm']).fillna(0).astype(float)
    adata_umap.obs['DPD_btla_norm'] = adata_umap.obs['perturbed_gene_name'].map(
        dpd_lookup['DPD_btla_norm']).fillna(0).astype(float)
    adata_umap.obs['is_ntc'] = adata_umap.obs['guide_type'] == 'non-targeting'

    # Plot 1: coloured by DPD_stim_norm
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    coords = adata_umap.obsm['X_umap']
    stim_vals = adata_umap.obs['DPD_stim_norm'].values
    btla_vals = adata_umap.obs['DPD_btla_norm'].values

    for ax, vals, label, cmap in [
        (axes[0], stim_vals, 'DPD_stim_norm', 'RdBu_r'),
        (axes[1], btla_vals, f'DPD_btla_norm ({BTLA_TIMEPOINT})', 'PuOr_r')
    ]:
        vmax = np.percentile(np.abs(vals[vals != 0]), 95) if (vals != 0).any() else 1
        sc_plot = ax.scatter(coords[:, 0], coords[:, 1],
                             c=vals, cmap=cmap, vmin=-vmax, vmax=vmax,
                             s=2, alpha=0.6, rasterized=True)
        plt.colorbar(sc_plot, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xlabel('UMAP 1', fontsize=11)
        ax.set_ylabel('UMAP 2', fontsize=11)
        ax.set_title(f'UMAP coloured by {label}', fontsize=12, fontweight='bold')

    plt.suptitle(f'Core gene perturbations: {CONDITION}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    p = os.path.join(OUT_DIR, f'umap_core_perturbations_{run_tag}.png')
    plt.savefig(p, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: {p}')

    # Plot 2: coloured by perturbation identity (highlight top 10 each tail)
    top10_act = core_df.nlargest(10, 'DPD_stim_norm')['target_contrast_gene_name'].tolist()
    top10_sup = core_df.nsmallest(10, 'DPD_stim_norm')['target_contrast_gene_name'].tolist()
    highlight = set(top10_act + top10_sup)
    label_col = adata_umap.obs['perturbed_gene_name'].apply(
        lambda x: x if x in highlight else ('NTC' if x == 'non-targeting' else 'other'))
    adata_umap.obs['highlight_label'] = label_col

    fig, ax = plt.subplots(figsize=(12, 9))
    palette = {'NTC': '#cccccc', 'other': '#dddddd'}
    for gene in top10_act:
        palette[gene] = '#d62728'
    for gene in top10_sup:
        palette[gene] = '#1f77b4'

    for label in ['NTC', 'other'] + top10_act + top10_sup:
        mask = adata_umap.obs['highlight_label'] == label
        size = 2 if label in ['NTC', 'other'] else 12
        alpha = 0.3 if label in ['NTC', 'other'] else 0.9
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=palette[label], s=size, alpha=alpha,
                   label=label if label not in ['NTC', 'other'] else None,
                   rasterized=True)
    ax.legend(markerscale=4, fontsize=8, ncol=2, loc='best')
    ax.set_xlabel('UMAP 1', fontsize=11)
    ax.set_ylabel('UMAP 2', fontsize=11)
    ax.set_title(f'Top 10 activators (red) and suppressors (blue), {CONDITION}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    p = os.path.join(OUT_DIR, f'umap_top_perturbations_labelled_{run_tag}.png')
    plt.savefig(p, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: {p}')

    del adata_umap
    gc.collect()

RUN_UMAP=False, skipping.


### 6. Build Cross-Perturbation Matrices

M[i,j] = e_i · e_j, the dot product of the full expression profiles of perturbations i and j. A high positive value means the two knockdowns produce similar transcriptional responses; negative means opposing responses.

The cosine-normalised version divides by the product of row L2 norms, removing differences in perturbation magnitude (how many genes change) so only the direction (which genes change and in what direction) is captured.

Both axes (stim and BTLA) use the same method (`expr @ expr.T`) operating on their respective expression matrices. This is methodologically consistent and contrasts with the original pipeline where the BTLA matrix was computed as an outer product of scalar DPD projections (which discarded all cross-gene structure).

In [9]:
def build_cross_pert(expr_matrix, label):
    raw = expr_matrix @ expr_matrix.T
    norms = np.linalg.norm(expr_matrix, axis=1, keepdims=True)
    norms = np.where(norms < 1e-12, 1e-12, norms)
    cosine = raw / (norms @ norms.T)
    print(f'{label} cross-pert: raw range [{raw.min():.3f}, {raw.max():.3f}]  '
          f'cosine range [{cosine.min():.3f}, {cosine.max():.3f}]')
    return raw, cosine

raw_stim, cosine_stim = build_cross_pert(expr_stim, 'stim')
raw_btla, cosine_btla = build_cross_pert(expr_btla, 'btla')

# Sanity check: DPD scores recomputed from expr_stim should correlate
# strongly with the original DPD_stim_norm values from Notebook 1.
# v_stim is already aligned to stim_genes_used (same order as stim_col_idx).
dpd_recomp = (expr_stim @ v_stim) / stim_norm
orig_vals = core_df.set_index('target_contrast_gene_name')['DPD_stim_norm'].reindex(core_genes).values
r = np.corrcoef(dpd_recomp, orig_vals)[0, 1]
print(f'\nDPD_stim correlation (recomputed vs Notebook 1): r={r:.4f}  (expect > 0.95)')
if r < 0.90:
    print('WARNING: low correlation, check that run_tag and CONDITION match Notebook 1')

stim cross-pert: raw range [-106.869, 454.426]  cosine range [-0.496, 1.000]
btla cross-pert: raw range [-1035.210, 4918.263]  cosine range [-0.403, 1.000]

DPD_stim correlation (recomputed vs Notebook 1): r=0.8679  (expect > 0.95)


### 7. Save

In [10]:
# Core gene annotation table (node table for Notebooks 3+)
core_path = os.path.join(OUT_DIR, f'core_genes_{run_tag}.csv')
core_df.to_csv(core_path, index=False)
print(f'Saved: {core_path}')

# Cross-perturbation matrices, stim axis
pd.DataFrame(raw_stim, index=core_genes, columns=core_genes).to_csv(
    os.path.join(OUT_DIR, f'cross_pert_raw_stim_{run_tag}.csv'))
pd.DataFrame(cosine_stim, index=core_genes, columns=core_genes).to_csv(
    os.path.join(OUT_DIR, f'cross_pert_cosine_stim_{run_tag}.csv'))
print(f'Saved: cross_pert_raw_stim and cross_pert_cosine_stim')

# Cross-perturbation matrices, btla axis
pd.DataFrame(raw_btla, index=core_genes, columns=core_genes).to_csv(
    os.path.join(OUT_DIR, f'cross_pert_raw_btla_{run_tag}.csv'))
pd.DataFrame(cosine_btla, index=core_genes, columns=core_genes).to_csv(
    os.path.join(OUT_DIR, f'cross_pert_cosine_btla_{run_tag}.csv'))
print(f'Saved: cross_pert_raw_btla and cross_pert_cosine_btla')

# Expression matrices (needed by Notebook 3 ROLS path as reference)
pd.DataFrame(expr_stim, index=core_genes, columns=stim_genes_used).to_csv(
    os.path.join(OUT_DIR, f'expr_matrix_stim_{run_tag}.csv'))
pd.DataFrame(expr_btla, index=core_genes, columns=gene_ids_de_stats).to_csv(
    os.path.join(OUT_DIR, f'expr_matrix_btla_{run_tag}.csv'))
print(f'Saved: expr_matrix_stim and expr_matrix_btla')

print(f'\nDone, {run_tag}')
print(f'Core genes: {len(core_genes)}')
print(f'Outputs in: {OUT_DIR}')

Saved: ../Results/Rest/core_genes_Rest_D1_D2_D3_D4.csv
Saved: cross_pert_raw_stim and cross_pert_cosine_stim
Saved: cross_pert_raw_btla and cross_pert_cosine_btla
Saved: expr_matrix_stim and expr_matrix_btla

Done, Rest_D1_D2_D3_D4
Core genes: 200
Outputs in: ../Results/Rest
